# Aula 06 — MCP (Model Context Protocol)

Nesta aula transformamos a *tool* de clima da [aula 04](aula_04_openai_functions.ipynb) em um
**servidor MCP** e depois consumimos esse servidor por um cliente e por um LLM.

MCP padroniza como um modelo descobre e chama ferramentas: o servidor anuncia `tools`,
`resources` e `prompts`; qualquer cliente compatível (este notebook, Claude Desktop, VS Code,
Cursor...) consegue usar sem código específico.

> **Atenção à versão do `mcp`.** A partir do `mcp` 2.0 o módulo `mcp.server.fastmcp` foi
> renomeado para `mcp.server.mcpserver` (e a classe `FastMCP` virou `MCPServer`). Este notebook
> usa a API `FastMCP`, então o `requirements.txt` fixa `mcp>=1.7.1,<2.0`. Se você vir
> `ModuleNotFoundError: No module named 'mcp.server.fastmcp'`, é porque o `mcp` 2.x foi instalado.

## 0 — Setup

In [1]:
import json
import os
from importlib.metadata import version

import requests
from dotenv import find_dotenv, load_dotenv

_ = load_dotenv(find_dotenv())  # le o .env local

OPENAI_API_KEY = os.environ["OPENAI_API_KEY"]
WEATHER_API_KEY = os.environ["WEATHER_API_KEY"]  # https://home.openweathermap.org/

print("mcp:", version("mcp"))  # precisa ser 1.x para `mcp.server.fastmcp`

mcp: 1.29.0


## 1 — O ponto de partida: a API de clima "na mão"

Antes do MCP, uma chamada direta à OpenWeatherMap. Note o parâmetro `units`: a API aceita
`standard` (Kelvin), `metric` (Celsius) ou `imperial` (Fahrenheit) — passar `"celsius"` faz a API
ignorar o valor e devolver Kelvin.

In [2]:
UNITS = {"kelvin": "standard", "celsius": "metric", "fahrenheit": "imperial"}

response = requests.get(
    "https://api.openweathermap.org/data/2.5/weather",
    params={"q": "Recife", "appid": WEATHER_API_KEY, "units": UNITS["celsius"]},
    timeout=10,
)
response.raise_for_status()
response.json()

{'coord': {'lon': -34.8811, 'lat': -8.0539},
 'weather': [{'id': 802,
   'main': 'Clouds',
   'description': 'scattered clouds',
   'icon': '03n'}],
 'base': 'stations',
 'main': {'temp': 26.02,
  'feels_like': 26.02,
  'temp_min': 26.02,
  'temp_max': 26.02,
  'pressure': 1017,
  'humidity': 69,
  'sea_level': 1017,
  'grnd_level': 1014},
 'visibility': 10000,
 'wind': {'speed': 5.14, 'deg': 150},
 'clouds': {'all': 47},
 'dt': 1787258441,
 'sys': {'type': 1,
  'id': 8426,
  'country': 'BR',
  'sunrise': 1787214410,
  'sunset': 1787257164},
 'timezone': -10800,
 'id': 3390760,
 'name': 'Recife',
 'cod': 200}

## 2 — Instalação

Já está no `requirements.txt`, mas se precisar instalar no ambiente atual:

In [3]:
# !pip install "mcp>=1.7.1,<2.0" uv

## 3 — O servidor MCP

O `FastMCP` cuida do protocolo; nós só decoramos funções Python:

- `@mcp.tool()` → ação que o modelo pode **executar** (a assinatura e a docstring viram o schema).
- `@mcp.resource()` → dado que o cliente pode **ler**, endereçado por URI.
- `@mcp.prompt()` → template de prompt reutilizável.

O servidor roda em um processo separado e conversa por `stdio`, então precisa carregar o `.env`
e importar tudo o que usa — ele não herda nada do notebook.

In [4]:
os.makedirs("mcp_demo", exist_ok=True)  # %%writefile nao cria diretorios

# Cuidado: nunca nomeie esta pasta como `mcp/` — ela viraria um pacote local
# que sombrearia a biblioteca `mcp` instalada, e o import passaria a falhar.

In [5]:
%%writefile mcp_demo/server.py
"""Servidor MCP de exemplo — expoe o clima da OpenWeatherMap."""

import os

import requests
from dotenv import find_dotenv, load_dotenv
from mcp.server.fastmcp import FastMCP

load_dotenv(find_dotenv())

WEATHER_API_KEY = os.environ["WEATHER_API_KEY"]
UNITS = {"kelvin": "standard", "celsius": "metric", "fahrenheit": "imperial"}

mcp = FastMCP("weather")


@mcp.tool()
def get_current_weather(location: str, unit: str = "celsius") -> dict:
    """Get the current weather in a given location."""
    if unit not in UNITS:
        raise ValueError(f"Invalid unit. Must be one of {list(UNITS)}.")

    response = requests.get(
        "https://api.openweathermap.org/data/2.5/weather",
        params={"q": location, "appid": WEATHER_API_KEY, "units": UNITS[unit]},
        timeout=10,
    )
    response.raise_for_status()
    data = response.json()

    return {
        "location": data["name"],
        "temperature": data["main"]["temp"],
        "unit": unit,
        "forecast": [data["weather"][0]["description"]],
    }


@mcp.resource("weather://units")
def list_units() -> str:
    """Unidades aceitas pela tool de clima."""
    return ", ".join(UNITS)


@mcp.prompt()
def mala_de_viagem(location: str) -> str:
    """Prompt pronto: o que levar na mala."""
    return f"Consulte o clima de {location} e diga o que devo levar na mala."


if __name__ == "__main__":
    # stdio: o cliente sobe este arquivo como subprocesso e fala pelo stdin/stdout.
    mcp.run(transport="stdio")

Overwriting mcp_demo/server.py


## 4 — Um cliente MCP

O cliente sobe o servidor como subprocesso, faz o *handshake* (`initialize`) e a partir daí
pergunta o que existe (`list_tools`) e executa (`call_tool`).

Como as conexões são *context managers* assíncronos, elas não podem ficar abertas entre células
do notebook. O helper abaixo abre a sessão, roda uma função e fecha.

In [6]:
from mcp import ClientSession, StdioServerParameters
from mcp.client.stdio import stdio_client

SERVER = StdioServerParameters(command="python", args=["mcp_demo/server.py"])


async def with_session(fn):
    """Sobe o servidor MCP, executa fn(session) e encerra o subprocesso."""
    async with stdio_client(SERVER) as (read, write):
        async with ClientSession(read, write) as session:
            await session.initialize()
            return await fn(session)

### 4.1 — O que o servidor oferece?

In [7]:
async def inspect(session):
    tools = await session.list_tools()
    resources = await session.list_resources()
    prompts = await session.list_prompts()
    return tools.tools, resources.resources, prompts.prompts


tools, resources, prompts = await with_session(inspect)

for tool in tools:
    print(f"TOOL     {tool.name}: {tool.description}")
    print(f"         schema: {json.dumps(tool.inputSchema)}")
for resource in resources:
    print(f"RESOURCE {resource.uri}: {resource.description}")
for prompt in prompts:
    print(f"PROMPT   {prompt.name}: {prompt.description}")

TOOL     get_current_weather: Get the current weather in a given location.
         schema: {"properties": {"location": {"title": "Location", "type": "string"}, "unit": {"default": "celsius", "title": "Unit", "type": "string"}}, "required": ["location"], "title": "get_current_weatherArguments", "type": "object"}
RESOURCE weather://units: Unidades aceitas pela tool de clima.
PROMPT   mala_de_viagem: Prompt pronto: o que levar na mala.


### 4.2 — Chamando a tool

In [8]:
result = await with_session(
    lambda session: session.call_tool(
        "get_current_weather", {"location": "Recife", "unit": "celsius"}
    )
)
print(result.content[0].text)

{
  "location": "Recife",
  "temperature": 26.02,
  "unit": "celsius",
  "forecast": [
    "scattered clouds"
  ]
}


## 5 — Ligando o servidor MCP ao LLM

O schema que o MCP entrega é o mesmo formato JSON Schema que o *function calling* da OpenAI
espera, então a ponte é quase mecânica: traduzir a lista de tools e, a cada `tool_call` do
modelo, encaminhar para `session.call_tool`.

A diferença em relação à aula 04: o notebook não conhece nenhuma função de clima. Ele só sabe
falar MCP — trocar de servidor troca as capacidades do agente sem mexer neste código.

In [9]:
from openai import OpenAI

client = OpenAI()
MODEL = "gpt-4o-mini"


def to_openai_tools(mcp_tools):
    """Converte tools MCP para o formato de function calling da OpenAI."""
    return [
        {
            "type": "function",
            "function": {
                "name": tool.name,
                "description": tool.description or "",
                "parameters": tool.inputSchema,
            },
        }
        for tool in mcp_tools
    ]


async def ask(question, verbose=True):
    async with stdio_client(SERVER) as (read, write):
        async with ClientSession(read, write) as session:
            await session.initialize()
            openai_tools = to_openai_tools((await session.list_tools()).tools)

            messages = [{"role": "user", "content": question}]

            while True:
                response = client.chat.completions.create(
                    model=MODEL, messages=messages, tools=openai_tools
                )
                message = response.choices[0].message
                messages.append(message)

                if not message.tool_calls:
                    return message.content

                for call in message.tool_calls:
                    args = json.loads(call.function.arguments)
                    if verbose:
                        print(f"-> {call.function.name}({args})")
                    tool_result = await session.call_tool(call.function.name, args)
                    messages.append(
                        {
                            "role": "tool",
                            "tool_call_id": call.id,
                            "content": tool_result.content[0].text,
                        }
                    )

In [10]:
print(await ask("Qual o clima em Recife agora? Preciso de guarda-chuva?"))

-> get_current_weather({'location': 'Recife'})


Atualmente, em Recife, a temperatura é de aproximadamente 26°C e o clima apresenta algumas nuvens dispersas. Não parece haver indicação de chuva, então você provavelmente não precisará de um guarda-chuva.


### 5.1 — Várias chamadas na mesma pergunta

In [11]:
print(await ask("Compare o clima de Recife, São Paulo e Lisboa. Onde está mais fresco?"))

-> get_current_weather({'location': 'Recife', 'unit': 'celsius'})


-> get_current_weather({'location': 'São Paulo', 'unit': 'celsius'})


-> get_current_weather({'location': 'Lisboa', 'unit': 'celsius'})


Atualmente, as temperaturas nas três cidades são as seguintes:

- **Recife**: 26,02°C com algumas nuvens dispersas.
- **São Paulo**: 29,43°C com céu limpo.
- **Lisboa**: 20,23°C com céu limpo.

Comparando as temperaturas, Lisboa está mais fresca, com 20,23°C.


## 6 — Usando o servidor fora do notebook

### MCP Inspector (interface web para testar o servidor)

Rode a partir de `books/`:

```bash
npx @modelcontextprotocol/inspector python mcp_demo/server.py
```

Ele imprime a URL com um token de sessão — **abra exatamente essa URL**, a porta é a **6274**
(a `6275` é só o sandbox de MCP Apps):

```
MCP Inspector Web is up and running at:
   http://localhost:6274?MCP_INSPECTOR_API_TOKEN=<token>
```

> **Rodando dentro do Docker?** Duas coisas precisam estar no lugar, e as duas já estão no
> `docker-compose.yml` deste repo:
>
> 1. As portas `6274` e `6275` publicadas para o host.
> 2. `HOST=0.0.0.0` **e** `DANGEROUSLY_BIND_ALL_INTERFACES=true`. Por padrão o Inspector escuta só
>    em `127.0.0.1` *dentro* do container, o que é inalcançável do host mesmo com a porta
>    publicada. E ele se recusa a abrir em todas as interfaces sem o override — a proteção existe
>    contra DNS rebinding, já que o backend do Inspector pode iniciar processos locais. Aqui é
>    seguro porque o compose publica as portas apenas no loopback do host (`127.0.0.1:6274`).
>
> Copie a URL do terminal para o navegador do host: o `Opening browser...` não tem como abrir nada
> dentro do container.

### Claude Desktop / VS Code / Cursor

Registre o servidor no arquivo de configuração do cliente (por exemplo
`claude_desktop_config.json`) e ele aparece como ferramenta disponível no chat:

In [12]:
config = {
    "mcpServers": {
        "weather": {
            "command": "python",
            "args": [os.path.abspath("mcp_demo/server.py")],
            "env": {"WEATHER_API_KEY": "sua-chave-aqui"},
        }
    }
}
print(json.dumps(config, indent=2))

{
  "mcpServers": {
    "weather": {
      "command": "python",
      "args": [
        "/app/books/mcp_demo/server.py"
      ],
      "env": {
        "WEATHER_API_KEY": "sua-chave-aqui"
      }
    }
  }
}


### 6.1 — Um terminal dentro do notebook (opcional)

O Inspector é um processo separado e de longa duração, então precisa de um shell. O caminho normal
é `make docker-start`, que abre um bash no container. Se preferir não sair do navegador, a célula
abaixo embute um terminal do próprio Jupyter.

Dois detalhes que fazem isso funcionar (e que quebram se você chutar a URL):

- A porta é descoberta do servidor em execução — neste container o Jupyter está na **8080**, não na
  8888.
- O `src` é **relativo**. O Jupyter responde `Content-Security-Policy: frame-ancestors 'self'`, ou
  seja, só aceita ser embutido pela própria origem; e um caminho relativo já vai autenticado pelo
  cookie da sessão, sem precisar passar token na URL.

In [13]:
import requests
from IPython.display import IFrame
from jupyter_server.serverapp import list_running_servers

server = next(iter(list_running_servers()))
base_url = server["base_url"]  # normalmente "/"

# O terminal precisa existir antes de ser exibido: `/terminals/1` nao cria nada.
terminal = requests.post(
    f"http://localhost:{server['port']}{base_url}api/terminals",
    params={"token": server["token"]},
    timeout=10,
).json()["name"]

print(f"terminal {terminal} — cwd do Jupyter: {server['root_dir']}")
IFrame(f"{base_url}terminals/{terminal}", width="100%", height=500)

terminal 3 — cwd do Jupyter: /app


### 6.2 — Conectando o Inspector ao servidor

Ao abrir a UI, o **Transport Type precisa estar em `STDIO`** — `Command: python`,
`Arguments: mcp_demo/server.py`. Se ele estiver em `SSE` ou `Streamable HTTP`, o Inspector trata o
comando como se fosse uma URL e tenta autenticar nele, resultando em:

```
OAuth authorization failed for "python"
OAuth is only supported for HTTP-based transports (SSE, streamable-http)
```

Para não depender de acertar os campos na mão, descreva o servidor num arquivo e passe com
`--config`. O Inspector já abre com a entrada pronta, no transporte certo:

In [14]:
%%writefile mcp_demo/inspector.json
{
  "mcpServers": {
    "weather": {
      "command": "python",
      "args": ["mcp_demo/server.py"]
    }
  }
}

Overwriting mcp_demo/inspector.json


```bash
# a partir de books/
npx @modelcontextprotocol/inspector --config mcp_demo/inspector.json --server weather
```

O mesmo arquivo serve para o modo `--cli`, útil para testar sem navegador nenhum — e aí o
`--server` é respeitado:

```bash
npx @modelcontextprotocol/inspector --cli --config mcp_demo/inspector.json --server weather \
    --method tools/list

npx @modelcontextprotocol/inspector --cli --config mcp_demo/inspector.json --server weather \
    --method tools/call --tool-name get_current_weather --tool-arg location=Recife
```

## Exercícios

1. Adicione uma tool `get_forecast(location, days)` usando o endpoint `/forecast` da
   OpenWeatherMap e confirme que ela aparece no `list_tools()` sem nenhuma mudança no cliente.
2. Faça a tool devolver erro amigável (em vez de exceção) quando a cidade não existir, e observe
   como o modelo reage ao erro.
3. Troque o transporte de `stdio` para `streamable-http` (`mcp.run(transport="streamable-http")`)
   e ajuste o cliente para `mcp.client.streamable_http`.
4. Exponha as ferramentas da [aula 05](aula_05_agents_langgraph.ipynb) (Wikipedia, arXiv) como um
   segundo servidor MCP e conecte os dois na mesma conversa.